# TaskB: Matching the Job-Title with the Relevant Skills with Contextual Representations

In this notebook, we prepare the shared-track Task-B dataset 
- for developing the job-title to Skills matching model and
- for evaluating the predicted matched skills using evaluation script [task's evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script).

- Additionally, the provided format is also compatible with the testset.

## Import Required Libraries

In [2]:
import os
import sys
import json

In [3]:
import pandas as pd
import numpy as np
import subprocess
import ast

In [4]:
from sentence_transformers import SentenceTransformer, util
from codecarbon import EmissionsTracker

## Matching the JobTitle with Skills 

### Setting up data path

In [5]:
root_data_path="taskB"

In [6]:
#Training data path
taskB_training_path = os.path.join(root_data_path, "training")

#validation data path
validation_data_path=os.path.join(root_data_path, "validation")

#testing data path
test_data_path=os.path.join(root_data_path, "test")

#model path
models_path = "models"

#results path
results_path = "results"

#evaluation result path
evaluations_path = "evaluations"

## Training Data

### Training data loading

#### Training data dictionary

The training data for Task B is shared in 3 files:
- `job2skill.tsv`: This file has been curated to include the most representative skills for each job title in <a href="https://esco.ec.europa.eu/en">ESCO</a>. This file contains three columns:
  - `job_id`: ESCO identifier for the job position.
  - `skill_id`: ESCO identifier for the skill.
  - `rel_type`: Indicator specifying whether the skill_id is essential or optional for a specific job_id. It can have the value “essential” or “optional.”

- `jobid2terms.json`: This JSON file contains job_id identifiers used in the training set for Task B as keys, and a list of valid lexical variants for each identifier as values.

- `skillid2terms.json`: This JSON file contains skill_id identifiers as keys, and a list of valid lexical variants for each identifier as values.


#### Reading Job2Skill data file

In [ ]:
# Read job2skill file
job2skill = pd.read_csv(os.path.join(taskB_training_path, 'job2skill.tsv'),
                        sep="\t",
                        names=["job_id","skill_id","rel_type"])

print ("No. of job_id and skill_id triple: {}".format(job2skill.shape[0]))
job2skill.head()

#### Reading Job2Terms file

In [ ]:
# Read json files
jobid2terms_path = os.path.join(taskB_training_path, "jobid2terms.json")
print (jobid2terms_path)

In [ ]:
with open(os.path.join(jobid2terms_path), 'r') as file:
    jobid2terms = json.load(file)

In [ ]:
print ("Total job ids: {}".format(len(jobid2terms)))

#### Reading Skilled2Terms

In [ ]:
# Read json files
skillid2terms_path = os.path.join(taskB_training_path, "skillid2terms.json")
print (skillid2terms_path)

In [ ]:
with open(os.path.join(skillid2terms_path), 'r') as file:
    skillid2terms = json.load(file)

In [ ]:
print ("Total skill ids: {}".format(len(skillid2terms)))

#### Play with Job-id, Skill-id, Job-id-terms, and Skill-id-terms. 

##### Expanding Job2Skill with Terms Mapping

In [ ]:
job2skill["job_terms"] = job2skill["job_id"].map(jobid2terms)
job2skill["skill_terms"] = job2skill["skill_id"].map(skillid2terms)

In [ ]:
job2skill.describe()

**Let's see the skills for a specific job_id.**



In [ ]:
job_id_to_search = job2skill.job_id.to_list()[100]
job_data = job2skill[job2skill.job_id == job_id_to_search]

In [ ]:
job_id_to_search

**The job titles of the selected job_id are:**

In [ ]:
job_data.job_terms.iloc[0]

**For that job_id we fin 9 essential skills and 1 optional skill:**

In [ ]:
job_data.rel_type.value_counts()

**The list of skills related to the job_id are:**

In [ ]:
job_data.skill_terms.to_list()[0]

In [ ]:
print ("JobID: {}\nSkillID: {}\nJobTerms: {}\nSkillTerms: {}".format(job2skill.job_id.to_list()[0], job2skill.skill_id.to_list()[0], job2skill.job_terms.to_list()[0], job2skill.skill_terms.to_list()[0]))

In [ ]:
job2skill.head(5)

## Validation Data

### **Validation data loading**

#### **Validation data path setup**

In [ ]:
validation_queries_path = os.path.join(validation_data_path, "queries") #queries
validation_corpus_elements_path = os.path.join(validation_data_path, "corpus_elements") #documents

#### **Load queries and corpus elements**

In [ ]:
validation_queries = pd.read_csv(validation_queries_path, sep="\t")
validation_corpus_elements = pd.read_csv(validation_corpus_elements_path, sep="\t")

In [ ]:
print ("Validation set: ")
print ("No. of queries: {}\nNo. of copurs elements: {}".format(len(validation_queries), len(validation_corpus_elements)))

In [ ]:
print ("validation query samples:")
validation_queries.head(10)

In [ ]:
print ("validation corpus elements samples:")
validation_corpus_elements.head(10)

In [ ]:
validation_corpus_elements.skill_aliases[0]

In [ ]:
validation_corpus_elements.esco_uri[0]

### **Matching JobTitle with Skills**

- *Idea 1: Adhoc (no machine learning)*
    - Preprocessing: Aggregate the skill aliases into one long text
    - Task: Match the job title text with the aggregated long skill text
        - Approach A: Traditional/Sparse matching model (TF-IDF, BM25, QLM)
            - Bag-of-words (BoWs) representation + TF-IDF model
            - BoWs representation + BM25 model
        - Approach B: Dense matching model (BERT, RoBERTa, GPT, SentenceTransformers)
            - Contextual representation + Cosine similarity

##### **Transform `skill_aliases` column to a list of strings**

In [ ]:
validation_corpus_elements["skill_aliases"] = validation_corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

In [ ]:
validation_corpus_elements.head()

##### **Aggregate the skill aliases into one long text**

In [17]:
def aggregate_list_of_terms(terms_list):
    '''
        aggregate the list of terms to a long text (string)
    '''
    return " ".join(terms_list)

In [ ]:
validation_corpus_elements["skill_document"] = validation_corpus_elements["skill_aliases"].apply(lambda x: aggregate_list_of_terms(x))

In [ ]:
validation_corpus_elements.head()

In [ ]:
validation_corpus_elements.skill_document[0]

#### Generating a mapping (dictionary) for queries

In [ ]:
'''
    build a dictionary/mapping between the query_id and text from the query pandas dataframe
'''
validation_queries_ids = validation_queries.q_id.to_list()
validation_queries_texts = validation_queries.jobtitle.to_list()

validation_queries_map = dict(zip(validation_queries_ids, validation_queries_texts))

In [ ]:
list(validation_queries_map.items())[:5]

In [ ]:
validation_corpus_elements.head(10)

#### Generaring a mapping for corpus elements

In [ ]:
'''
    build a dictionary/mapping between the corpus_id and document from the pandas dataframe
    Documents and elements are interchangably used
'''
validation_documents_ids = validation_corpus_elements.c_id.to_list()
#validation_documents_esco_uri = validation_corpus_elements.esco_uri.to_list()
validation_documents_texts = validation_corpus_elements.skill_document.to_list()

validation_documents_map = dict(zip(validation_documents_ids, validation_documents_texts))

In [ ]:
list(validation_documents_map.items())[:5]

#### **Loading Contextual Embedding model**

In [25]:
model = SentenceTransformer("all-MiniLM-L6-v2", token=False)

/home/zia/miniforge3/envs/pumpkin/lib/python3.10/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [26]:
#tracker = EmissionsTracker()
#tracker.start_task("all-MiniLM-L6-v2")

In [27]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

#### **Embedding Queries and Documents**
- Dense representation using SentenceTransformers

In [ ]:
validation_query_embeddings = model.encode(validation_queries_texts, convert_to_tensor=True)

In [ ]:
validation_query_embeddings.shape

In [ ]:
validation_query_embeddings[0]

In [ ]:
validation_documents_embeddings = model.encode(validation_documents_texts, convert_to_tensor=True)

In [ ]:
validation_documents_embeddings.shape 

In [ ]:
validation_documents_embeddings[0]

##### Cosine Similarity between Embedding Representation

In [ ]:
validation_similarities_query_documents = util.cos_sim(validation_query_embeddings, validation_documents_embeddings).cpu().numpy()

In [ ]:
validation_similarities_query_documents.shape

In [ ]:
validation_similarities_query_documents[0,:]

## Evaluation Preparation
- To compute the evaluation metrics, we will generate a submission file using the query-documents similarity metric.
- The submission file must follow the ``TREC Run File format'', including headers in the output file.
- This means that the file should have 6 space-spearated columns per line, with following information:
    - q_id: query ID.
    - Q0: a constant identifier, usually "Q0".
    - doc_id: id of the retrieved corpus elements (skill_id).
    - rank: position of the document in the ranking.
    - score: relevance score assigned by the model.
    - tag: experiment name

### Preparing Submission File

In [34]:
def get_ranked_result_list(queries_ids, queries_map, documents_ids, documents_map, similarities_query_documents, model_name):
    results = []
    results_name = []
    
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities_query_documents[q_idx])
        used_doc_ids = set()
        rank_counter = 0
        for c_idx in sorted_indices:  # Consider the full list.
            doc_id = documents_ids[c_idx]
            # If doc_id was already processed, go to the next one.
            if doc_id in used_doc_ids:
                continue
            used_doc_ids.add(doc_id)
            rank_counter += 1
    
            query_name = queries_map[q_id]
            doc_name = documents_map[doc_id]
            score = similarities_query_documents[q_idx, c_idx]
    
            results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} {model_name}")
            results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} {model_name}")

    return results, results_name

In [35]:
validation_results_list, validation_results_name = get_ranked_result_list(validation_queries_ids, validation_queries_map,
                                                    validation_documents_ids, validation_documents_map,
                                                    validation_similarities_query_documents,
                                                   "Adhoc_ST_Cosine")

NameError: name 'validation_queries_ids' is not defined

In [ ]:
validation_results_list[:5], validation_results_name[:5]

In [ ]:
results_path

In [ ]:
validation_result_adhoc_STC_taskB = os.path.join(results_path, "validation_adhoc_sentencetransformer_taskB.trec")
with open(validation_result_adhoc_STC_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(validation_results_list))

### Evaluation Scripts

- For the evaluation, we will use the official [TalentCLEF evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script), which uses the Ranx library under the hood.

- First, clone the repo and install the requirements file:

In [ ]:
# run this cell just once for the first time
#!git clone https://github.com/TalentCLEF/talentclef25_evaluation_script.git
#!pip install -r talentclef25_evaluation_script/requirements.txt

Then, select the Qrels file and the Run file to perform the evaluation.

In [ ]:
qrels_file = os.path.join(validation_data_path, "qrels.tsv")
run_validation_adhoc_STC_file = os.path.join(results_path, "validation_adhoc_sentencetransformer_taskB.trec")

In [ ]:
qrels_file, run_validation_adhoc_STC_file

In [ ]:
print ("Evaluation")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_validation_adhoc_STC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Alternative approach for computing evaluation metrics

!python talentclef25_evaluation_script/talentclef_evaluate.py --qrels taskB/validation/qrels.tsv --run results/validation_adhoc_sentencetransformer_taskB.trec

## Testing Data

### Testing data loading

#### Testing data path setup

In [7]:
test_queries_path = os.path.join(test_data_path, "queries") #queries
test_corpus_elements_path = os.path.join(test_data_path, "corpus_elements") #documents

#### Loading query and corpus elements

In [8]:
test_queries = pd.read_csv(test_queries_path, sep="\t")
test_corpus_elements = pd.read_csv(test_corpus_elements_path, sep="\t")

#### Data explorations

In [9]:
print ("Test set: ")
print ("No. of queries: {}\nNo. of copurs elements: {}".format(len(test_queries), len(test_corpus_elements)))

Test set: 
No. of queries: 1520
No. of copurs elements: 1986


In [10]:
print ("test query samples:")
test_queries.head(10)

test query samples:


,q_id,jobtitle
0,15ec5e80,Qdoba Mexican Eats
1,a0d97669,Hairstylist
2,4b3976f7,Digital Logic
3,a3bc11cc,Interstate/Local Moving Contractor
4,e12ee1fd,Press Operator
5,a3bc11cc,Interstate/Local Moving Contractor
6,d98a5488,Case Goods Assembler
7,860893f9,Access Controller
8,c9565a52,Nucleus Rtos
9,7f0fbaa4,Thermoplastic Fabricator


In [12]:
print ("test corpus elements samples:")
test_corpus_elements.head(10)

test corpus elements samples:


,c_id,esco_uri,skill_aliases
0,f3a278e7,http://data.europa.eu/esco/skill/77e41477-4ecf...,['recruiting and hiring']
1,2938fcd6,http://data.europa.eu/esco/skill/450de541-797a...,"['sales activities', 'sales exercises', 'a sal..."
2,7b3c55d4,http://data.europa.eu/esco/skill/0fbaceff-df05...,"['carry out job analysis', 'carrying out job a..."
3,572ec5d6,http://data.europa.eu/esco/skill/714eebcb-569a...,"['develop staff', 'help strengthen staff', 'bu..."
4,73f0560b,http://data.europa.eu/esco/skill/5aa57bdc-5765...,"['hire human resources', 'hire experienced peo..."
5,4a7d2644,http://data.europa.eu/esco/skill/b3d9442d-1fb8...,"['write job descriptions', 'come up with job d..."
6,7e45cc8b,http://data.europa.eu/esco/skill/fbdbed1c-442a...,"['ergonomics', 'ergonomy', 'human factors and ..."
7,099a9ccf,http://data.europa.eu/esco/skill/9d9ba780-4aa2...,"['introduce new employees', 'introduce new emp..."
8,28bdf20b,http://data.europa.eu/esco/skill/88b406d0-72e2...,"['personnel management', 'human resource overs..."
9,d9acfa1a,http://data.europa.eu/esco/skill/21d2f96d-35f7...,"['computer programming', 'AngularJS', 'AWK', '..."


In [13]:
test_corpus_elements.skill_aliases[0]

"['recruiting and hiring']"

##### From Skill-aliases to list of terms

In [14]:
test_corpus_elements["skill_aliases"] = test_corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

In [15]:
test_corpus_elements.head()

,c_id,esco_uri,skill_aliases
0,f3a278e7,http://data.europa.eu/esco/skill/77e41477-4ecf...,[recruiting and hiring]
1,2938fcd6,http://data.europa.eu/esco/skill/450de541-797a...,"[sales activities, sales exercises, a sales ac..."
2,7b3c55d4,http://data.europa.eu/esco/skill/0fbaceff-df05...,"[carry out job analysis, carrying out job anal..."
3,572ec5d6,http://data.europa.eu/esco/skill/714eebcb-569a...,"[develop staff, help strengthen staff, build u..."
4,73f0560b,http://data.europa.eu/esco/skill/5aa57bdc-5765...,"[hire human resources, hire experienced people..."


##### Aggregate list of terms to text

In [18]:
test_corpus_elements["skill_document"] = test_corpus_elements["skill_aliases"].apply(lambda x: aggregate_list_of_terms(x))

In [19]:
test_corpus_elements

,c_id,esco_uri,skill_aliases,skill_document
0,f3a278e7,http://data.europa.eu/esco/skill/77e41477-4ecf...,[recruiting and hiring],recruiting and hiring
1,2938fcd6,http://data.europa.eu/esco/skill/450de541-797a...,"[sales activities, sales exercises, a sales ac...",sales activities sales exercises a sales activ...
2,7b3c55d4,http://data.europa.eu/esco/skill/0fbaceff-df05...,"[carry out job analysis, carrying out job anal...",carry out job analysis carrying out job analys...
3,572ec5d6,http://data.europa.eu/esco/skill/714eebcb-569a...,"[develop staff, help strengthen staff, build u...",develop staff help strengthen staff build up s...
4,73f0560b,http://data.europa.eu/esco/skill/5aa57bdc-5765...,"[hire human resources, hire experienced people...",hire human resources hire experienced people m...
...,...,...,...,...
1981,0a854753,http://data.europa.eu/esco/skill/4a1d8069-717c...,"[lead hiking trips, organise hiking trips, lea...",lead hiking trips organise hiking trips leads ...
1982,4cb25e2d,http://data.europa.eu/esco/skill/77e1563f-84fb...,"[cadastral taxation, real estate taxes, real e...",cadastral taxation real estate taxes real esta...
1983,7ab457c7,http://data.europa.eu/esco/isced-f/0221,[religion and theology],religion and theology
1984,7f3178fa,http://data.europa.eu/esco/skill/30385f84-3f06...,"[types of wine, type of wine, brands of wine, ...",types of wine type of wine brands of wine wine...


,c_id,esco_uri,skill_aliases,skill_document
0,f3a278e7,http://data.europa.eu/esco/skill/77e41477-4ecf...,[recruiting and hiring],recruiting and hiring
1,2938fcd6,http://data.europa.eu/esco/skill/450de541-797a...,"[sales activities, sales exercises, a sales ac...",sales activities sales exercises a sales activ...
2,7b3c55d4,http://data.europa.eu/esco/skill/0fbaceff-df05...,"[carry out job analysis, carrying out job anal...",carry out job analysis carrying out job analys...
3,572ec5d6,http://data.europa.eu/esco/skill/714eebcb-569a...,"[develop staff, help strengthen staff, build u...",develop staff help strengthen staff build up s...
4,73f0560b,http://data.europa.eu/esco/skill/5aa57bdc-5765...,"[hire human resources, hire experienced people...",hire human resources hire experienced people m...


#### Generate a mapping for test queries

In [22]:
'''
    build a dictionary/mapping between the query_id and text from the query pandas dataframe
'''
test_queries_ids = test_queries.q_id.to_list()
test_queries_texts = test_queries.jobtitle.to_list()

test_queries_map = dict(zip(test_queries_ids, test_queries_texts))

#### Generate a mapping for test corpus elements

In [23]:
'''
    build a dictionary/mapping between the corpus_id and document from the pandas dataframe
    Documents and elements are interchangably used
'''
test_documents_ids = test_corpus_elements.c_id.to_list()
#validation_documents_esco_uri = validation_corpus_elements.esco_uri.to_list()
test_documents_texts = test_corpus_elements.skill_document.to_list()

test_documents_map = dict(zip(test_documents_ids, test_documents_texts))

#### Embedding Queries and Documents

In [28]:
test_query_embeddings = model.encode(test_queries_texts, convert_to_tensor=True)

In [29]:
test_documents_embeddings = model.encode(test_documents_texts, convert_to_tensor=True)

In [30]:
test_query_embeddings.shape, test_documents_embeddings.shape

(torch.Size([1520, 384]), torch.Size([1986, 384]))

In [31]:
test_similarities_query_documents = util.cos_sim(test_query_embeddings, test_documents_embeddings).cpu().numpy()

In [32]:
test_similarities_query_documents

array([[ 0.0222042 , -0.01640725, -0.03812021, ...,  0.02827976,
        -0.04907396,  0.03896256],
       [ 0.16900167,  0.14369947,  0.09705906, ..., -0.00612994,
        -0.00859228,  0.00742529],
       [ 0.14036112,  0.13293807,  0.07486707, ...,  0.16595216,
         0.00384243,  0.08612566],
       ...,
       [ 0.31575137,  0.10840141,  0.1559203 , ..., -0.03736099,
        -0.04913865,  0.02670896],
       [ 0.26920706, -0.00272914,  0.25822762, ..., -0.00242457,
        -0.10783096,  0.01123484],
       [ 0.16985509,  0.06033069, -0.00294484, ...,  0.21048327,
         0.02376635,  0.13635008]], dtype=float32)

In [36]:
test_results_list, test_results_name = get_ranked_result_list(test_queries_ids, 
                                                              test_queries_map,
                                                              test_documents_ids, 
                                                              test_documents_map,
                                                              test_similarities_query_documents,
                                                              "adhoc_embedding_cosine")

In [37]:
test_results_list[:5], test_results_name[:5]

(['15ec5e80 Q0 89a6b046 1 0.3193 adhoc_embedding_cosine',
  '15ec5e80 Q0 241717a0 2 0.2977 adhoc_embedding_cosine',
  '15ec5e80 Q0 5f0ede54 3 0.2545 adhoc_embedding_cosine',
  '15ec5e80 Q0 93045d39 4 0.2470 adhoc_embedding_cosine',
  '15ec5e80 Q0 e645d5f9 5 0.2234 adhoc_embedding_cosine'],
 ['Qdoba Mexican Eats Q0 Spanish Castilian 1 0.3193 adhoc_embedding_cosine',
  'Qdoba Mexican Eats Q0 Portuguese 2 0.2977 adhoc_embedding_cosine',
  'Qdoba Mexican Eats Q0 nutrition diet 3 0.2545 adhoc_embedding_cosine',
  'Qdoba Mexican Eats Q0 Taleo 4 0.2470 adhoc_embedding_cosine',
  'Qdoba Mexican Eats Q0 prepare ready-made dishes prepare snacks and sandwiches prepare dishes that are ready-made prepare frozen food prepare catering dishes prepare fast food 5 0.2234 adhoc_embedding_cosine'])

In [38]:
test_result_adhoc_STC_taskB = os.path.join(results_path, "test_adhoc_sentencetransformer_taskB.trec")
with open(test_result_adhoc_STC_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(test_results_list))

In [43]:
test_qrels_file = os.path.join(test_data_path, "qrels.tsv")
test_run_adhoc_STC_file = os.path.join(results_path, "test_adhoc_sentencetransformer_taskB.trec")

In [44]:
test_qrels_file, test_run_adhoc_STC_file

('taskB/test/qrels.tsv', 'results/test_adhoc_sentencetransformer_taskB.trec')

In [45]:
print ("Evaluation")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", test_run_adhoc_STC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

Evaluation
Received parameters:
  qrels: taskB/test/qrels.tsv
  run: results/test_adhoc_sentencetransformer_taskB.trec
Loading qrels...

